# Lab 5: Transformers

The purpose of this notebook is to learn how to write a transformer from scratch. You will implement a decoder-type (e.g., GPT) framework for generating text.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and write code to complete the functions.

Note that you will need to connect your notebook to a GPU to initialise the model. You can do this by following the instructions in Lab 1.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from dataclasses import dataclass

Get dataset, here we'll use "TinyWikipedia".

In [ ]:
!wget https://sussex.box.com/shared/static/jkqin076fj6v2rts3tzv70il79rtvd4c.txt -O tiny_wikipedia.txt

In [ ]:
with open('tiny_wikipedia.txt', 'r', encoding='utf-8') as f:
    text = f.read()

Pre-process data (c.f. Week 4 lab).

In [ ]:
# Step 1) Get a sorted list of all unique characters that occur in this text
# Hint: set is useful for getting unique elements in a sequence
chars = sorted(list(set(text)))

# Step 2) Create the dictionaries str_to_int and int_to_str
str_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_str = {i: ch for i, ch in enumerate(chars)}

# Step 3) Define encode and decode functions
def encode(text, str_to_int):
    ids = [str_to_int[c] for c in text]
    return ids

def decode(ids, int_to_str):
    text_list = [int_to_str[id] for id in ids]
    return ''.join(text_list)

# Step 4) Test your implementation on "My dog Leo is extremely cute."
input_text = "My dog Leo is extremely cute."
ids = encode(input_text, str_to_int)
decoded_text = decode(ids, int_to_str)
assert input_text == decoded_text

Split your data and batch.

In [ ]:
# Train and test splits
data = torch.tensor(encode(text, str_to_int), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split, ctx_len, batch_size, device='cpu'):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - ctx_len, (batch_size,))
    x = torch.stack([data[i:i+ctx_len] for i in ix])
    y = torch.stack([data[i+1:i+ctx_len+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

Hyperparameters!

In [ ]:
@dataclass
class Config:
    d_model: int = 256 # the model/hidden/embedding dim
    n_heads: int = 4 # number of attention heads (width)
    ctx_len: int = 64 # context length
    batch_size: int = 8 # batch size
    n_layers: int = 12 # number of layers (depth)
    vocab_size: int = -1 # vocab size, to be determined once we have created a tokenizer
    device: str = 'cpu'

    def set_vocab_size(self, vocab_size):
        self.vocab_size = vocab_size

In [ ]:
config = Config()
config.set_vocab_size(vocab_size=len(chars)) # set our vocabular size (equal to the number of chars)

Define your embedding layer.

In [ ]:
class EmbeddingLayer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.wte = nn.Embedding(config.vocab_size, config.d_model)
        self.wpe = nn.Embedding(config.ctx_len, config.d_model)
        self.device = config.device

    def forward(self, x):
        B, T = x.shape

        x_tok = self.wte(x)
        # print(x_tok.shape) uncomment this if you want to see the shape of above tensor
        x_pos = self.wpe(torch.arange(T, device=self.device))
        # print(x_pos.shape)
        x_embeddings = x_tok + x_pos

        return x_embeddings


# Testing your implementation
xb, yb = get_batch('train', config.ctx_len, config.batch_size, config.device)

embedding = EmbeddingLayer(config)
x_embedding = embedding(xb)

assert x_embedding.shape == (config.batch_size, config.ctx_len, config.d_model), "Embedding dimensions are incorrect"

Define single headed attention.

In [ ]:
class SingleHeadCausalAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.head_dim = config.d_model // config.n_heads
        self.key = nn.Linear(config.d_model, self.head_dim, bias=False)
        self.query = nn.Linear(config.d_model, self.head_dim, bias=False)
        self.values = nn.Linear(config.d_model, self.head_dim, bias=False)

        self.register_buffer("cmask", torch.tril(torch.ones([config.ctx_len, config.ctx_len])))


    def forward(self, x):

        B, T, C = x.shape

        K = #TODO # (B, T, C) @ (_, C, H) -> (B, T, H)
        Q = #TODO
        V = #TODO

        y = #TODO # (B, T, H) @ (B, H, T) -> (B, T, T)
        y = torch.masked_fill(y, self.cmask[:T, :T]==0, float('-inf'))
        y = F.softmax(y, dim=-1) @ V
        return y


# Test your implementation
config = Config(d_model=256, n_heads=8, ctx_len=16)
attention = SingleHeadCausalAttention(config)
x = torch.randn(2, 10, 256)  # (batch_size, seq_len, d_model)
output = attention(x)
assert output.shape == (2, 10, 32)  # head_dim = 256/8 = 32

Define multi-head attention.

In [ ]:
class MultiHeadCausalAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList([SingleHeadCausalAttention(config) for _ in range(config.n_heads)])
        self.linear = nn.Linear(config.d_model, config.d_model)


    def forward(self, x):
        y = torch.cat([h(x) for h in self.heads], dim=-1)
        y = self.linear(y)
        return y


# Testing your implementation
config = Config(d_model=256, n_heads=8, ctx_len=16)
mha = MultiHeadCausalAttention(config)

# Test with small batch
x = torch.randn(2, 10, 256)  # (batch_size=2, seq_len=10, d_model=256)
out = mha(x)
assert out.shape == (2, 10, 256)

Define neural network block.

In [ ]:
class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.l1 = nn.Linear(config.d_model, 4*config.d_model)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(4*config.d_model, config.d_model)

    def forward(self, x):
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)
        return x

Define decoder block.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.mha = MultiHeadCausalAttention(config)
        self.ln1 = nn.LayerNorm(config.d_model)
        self.ffn = FFN(config)
        self.ln2 = nn.LayerNorm(config.d_model)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


# Testing your implementation
config = Config(d_model=256)
ffn = FFN(config)
decoder = DecoderBlock(config)

# Test with random input
x = torch.randn(2, 10, 256)  # (batch_size, sequence_length, d_model)
output = decoder(x)
assert output.shape == x.shape

Define decoder.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.Sequential(*[DecoderBlock(config) for _ in range(config.n_layers)])
        self.ln = nn.LayerNorm(config.d_model)
        self.lin = nn.Linear(config.d_model, config.vocab_size)
        self.emb = EmbeddingLayer(config)
        self.L = #TODO
        self.ctx_len = config.ctx_len
        self.device = config.device

    def forward(self, x, targets=None):
        B, T = x.shape
        x = self.emb(x)

        x = self.blocks(x)
        x = self.ln(x)
        logits = self.lin(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            # compute cross-entropy loss, targets are (B, T)
            B, T, V = logits.shape
            targets = targets.view(B*T)
            logits = logits.view(B*T, V)
            loss = self.L(logits, targets)

        return logits, loss

    def generate(self, idx, max_len=256):
        for _ in range(max_len):
            idx_window = idx[:, -self.ctx_len:]
            logits, _ = self(idx_window) #(B, T, V)
            logits = logits[:,-1,:]
            prob = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(prob, num_samples=1) # greedy sample
            idx = torch.cat((idx, next_token), dim=1)

        return idx


# Testing your implementation
config = Config(
    vocab_size=100,
    d_model=256,
    ctx_len=64,
    n_layers=4
)
decoder = Decoder(config)

x = torch.randint(0, 100, (1, 10))
logits, loss = decoder(x, x)

out = decoder.generate(torch.tensor([[1, 2, 3]]), max_len=5)
print(out.shape)  # Should be (1, 8) - original 3 tokens + 5 new ones

Set environment variables for CUDA support.

In [ ]:
import os
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:32"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print(os.environ["PYTORCH_CUDA_ALLOC_CONF"])
torch.cuda.empty_cache()

Train your transformer!

In [ ]:
# we're limited here by the GPU in the size of model we can have
config = Config(d_model=384, n_heads=12, ctx_len=256, batch_size = 64, n_layers = 12, device='cuda:0')
config.set_vocab_size(vocab_size=len(chars))
model = Decoder(config).to(config.device)

# print the size of the model
n_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {n_params}")

# more hyperparameters
learning_rate = 3e-4
max_iters = 6000
eval_interval = 200  # How often to evaluate
eval_iters = 100     # How many batches to use for evaluation

# Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    xb, yb = get_batch('train', config.ctx_len, config.batch_size, config.device)

    # forward pass
    logits, loss = model(xb, yb)

    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # evaluate on val data at specified intervals
    if iter % eval_interval == 0:
        model.eval()
        with torch.no_grad():
            val_losses = []
            for _ in range(eval_iters):
                xb, yb = get_batch('val', config.ctx_len, config.batch_size, config.device)
                _, val_loss = model(xb, yb)
                val_losses.append(val_loss.item())
            val_loss = sum(val_losses) / len(val_losses)

            print(f"step {iter}: train loss {loss.item():.4f}, val loss {val_loss:.4f}")

            # generate some text to see how we're doing
            context = torch.zeros((1, 1), dtype=torch.long, device=config.device)
            print(decode(model.generate(context, max_len=100)[0].tolist(), int_to_str))
            print('='*40)
        model.train()

# Final generation
context = torch.zeros((1, 1), dtype=torch.long, device=config.device)
print("\nFinal generated text:")
print(decode(model.generate(context, max_len=500)[0].tolist(), int_to_str))

**Extension**

Try nanoGPT, a fun ~300 line implementation of a transformer-based LLM!

https://github.com/karpathy/nanogpt